In [1]:
from pathlib import Path
import sys

import pandas as pd
from tqdm import tqdm
import fastf1
import fastf1.core
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

from f1winnerprediction import config, io_fastf1, mapper

pd.set_option("display.max_columns", None)

# expected_python = (config.PROJECT_ROOT / ".venv/bin/python").resolve()
# python_executable = Path(sys.executable).resolve()
# print(f"Python executable: {python_executable}")
# if expected_python.exists() and python_executable != expected_python:
#     raise EnvironmentError(
#         f"Activate the Poetry environment located at {expected_python} before running this notebook."
#     )

fastf1.Cache.enable_cache(config.FASTF1_RAW_CACHE_DIR.as_posix())

INTERIM_DIR = config.FASTF1_INTERIM_DATA_DIR


## Load cached sessions and GP mapping


In [2]:
sessions: dict[int, list[fastf1.core.Session]] = io_fastf1.load_sessions_from_years()
total_sessions = sum(len(year_sessions) for year_sessions in sessions.values())
print(f"Loaded {len(sessions)} seasons with {total_sessions} race sessions from disk.")

gp_index_map: dict[str, dict[int, int]] = mapper.map_gp_indices_between_years(sessions)
print(f"Computed GP mappings for {len(gp_index_map)} season pairs.")


Loaded 5 seasons with 114 race sessions from disk.
Computed GP mappings for 4 season pairs.


## Build consecutive-year lap time dataset


In [32]:
import importlib
importlib.reload(io_fastf1)

<module 'f1winnerprediction.io_fastf1' from '/media/dhiabenhamouda/Dhia/Work/F1WinnerPrediction/src/f1winnerprediction/io_fastf1.py'>

In [33]:
lap_time_pairs = io_fastf1.build_consecutive_lap_time_dataset(sessions, gp_index_map)
print(f"Laptime pairs shape: {lap_time_pairs.shape}")
lap_time_pairs.head()


Seasons: 100%|██████████| 5/5 [00:01<00:00,  4.54it/s]

Error processing seasons 2024 and 2025: The data you are trying to access has not been loaded yet. See `Session.load`
Laptime pairs shape: (1244, 6)


,Driver,PrevLapTimeSeconds,CurrentLapTimeSeconds,EventName,PrevSeason,CurrentSeason
0,ALO,102.290452,103.087263,Bahrain Grand Prix,2021,2022
1,BOT,98.310564,102.977246,Bahrain Grand Prix,2021,2022
2,GAS,100.181490,100.781023,Bahrain Grand Prix,2021,2022
3,HAM,97.586200,102.864193,Bahrain Grand Prix,2021,2022
4,LAT,102.558608,103.778579,Bahrain Grand Prix,2021,2022


## Load qualifying results from interim data


In [5]:
def load_qualifying_history(path: Path) -> pd.DataFrame:
    # Convert the wide qualifying dataset into a tidy format for easier joins.
    if not path.exists():
        raise FileNotFoundError(f"Qualifying file not found: {path}")
    wide_df = pd.read_csv(path)
    long_frames: list[pd.DataFrame] = []
    for column in wide_df.columns:
        if column == "Abbreviation":
            continue
        try:
            event_name, season_str, metric = column.rsplit("_", 2)
        except ValueError:
            continue
        if metric != "GridPosition":
            continue
        tmp = wide_df[["Abbreviation", column]].dropna().copy()
        tmp.rename(columns={"Abbreviation": "Driver", column: "GridPosition"}, inplace=True)
        tmp["EventName"] = event_name
        tmp["Season"] = int(season_str)
        long_frames.append(tmp)
    if not long_frames:
        return pd.DataFrame(columns=["Driver", "GridPosition", "EventName", "Season"])
    qualifying_df = pd.concat(long_frames, ignore_index=True)
    qualifying_df["GridPosition"] = qualifying_df["GridPosition"].astype(int)
    qualifying_df = qualifying_df[qualifying_df["GridPosition"] > 0]
    return qualifying_df


qualifying_path = INTERIM_DIR / "df_qualionly.csv"
qualifying_results = load_qualifying_history(qualifying_path)
print(f"Qualifying rows: {qualifying_results.shape[0]}")
qualifying_results.head()


Qualifying rows: 2170


,Driver,GridPosition,EventName,Season
0,NOR,7,Bahrain Grand Prix,2021
1,BOT,3,Bahrain Grand Prix,2021
3,MSC,18,Bahrain Grand Prix,2021
4,VER,1,Bahrain Grand Prix,2021
5,GIO,12,Bahrain Grand Prix,2021


## Merge lap times and qualifying features


In [6]:
merged_dataset = lap_time_pairs.merge(
    qualifying_results,
    how="inner",
    left_on=["Driver", "EventName", "CurrentSeason"],
    right_on=["Driver", "EventName", "Season"],
)
merged_dataset = merged_dataset.rename(columns={"GridPosition": "CurrentGridPosition"})
merged_dataset = merged_dataset.drop(columns=["Season"])
merged_dataset = merged_dataset[merged_dataset["CurrentGridPosition"] > 0].reset_index(drop=True)

MERGED_DATASET_PATH = INTERIM_DIR / "quali_lap_time_dataset.csv"
merged_dataset.to_csv(MERGED_DATASET_PATH, index=False)
print(f"Merged dataset saved to {MERGED_DATASET_PATH}")
print(f"Merged dataset shape: {merged_dataset.shape}")
merged_dataset.head()


Merged dataset saved to /media/dhiabenhamouda/Dhia/Work/F1WinnerPrediction/data/interim/quali_lap_time_dataset.csv
Merged dataset shape: (1235, 7)


,Driver,PrevLapTimeSeconds,CurrentLapTimeSeconds,EventName,PrevSeason,CurrentSeason,CurrentGridPosition
0,ALO,102.290452,103.087263,Bahrain Grand Prix,2021,2022,8
1,BOT,98.310564,102.977246,Bahrain Grand Prix,2021,2022,6
2,GAS,100.181490,100.781023,Bahrain Grand Prix,2021,2022,10
3,HAM,97.586200,102.864193,Bahrain Grand Prix,2021,2022,5
4,LAT,102.558608,103.778579,Bahrain Grand Prix,2021,2022,20


## Prepare features and hold-out split


In [7]:
feature_columns = ["PrevLapTimeSeconds", "CurrentGridPosition"]
target_column = "CurrentLapTimeSeconds"

X = merged_dataset[feature_columns].copy()
y = merged_dataset[target_column].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)
X_train.shape, X_test.shape


((988, 2), (247, 2))

## Train XGBoost regressor


In [8]:
xgb_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
}
regressor = xgb.XGBRegressor(
    objective="reg:squarederror",
    eval_metric="mae",
    random_state=42,
    **xgb_params,
)
regressor.fit(X_train, y_train)
regressor


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.9
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mae'


In [10]:
X_train

,PrevLapTimeSeconds,CurrentGridPosition
869,91.910161,5
728,74.002514,7
803,84.497386,5
644,110.378235,17
813,87.596804,11
...,...,...
1044,81.910413,12
1095,73.006543,11
1130,85.142357,6
860,101.432571,10


In [29]:
type(y_test), type(y_pred)

(pandas.core.series.Series, numpy.ndarray)

In [27]:
pred_vs_true = pd.DataFrame(
	{"ActualLapTime": y_test, "PredictedLapTime": y_pred},
	index=y_test.index,
)
pred_vs_true

,ActualLapTime,PredictedLapTime
753,100.030700,95.823311
582,97.424561,100.406631
548,78.438971,82.562202
113,98.378100,96.652092
174,91.186623,86.240143
...,...,...
31,89.958800,86.322250
778,109.035591,99.536751
910,86.131803,87.082535
361,107.282449,108.969688


## Evaluate model performance


In [14]:
y_pred = regressor.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Absolute Error: {mae:.4f} seconds")
print(f"R^2 Score: {r2:.4f}")

evaluation_sample = merged_dataset.loc[X_test.index, ["Driver", "EventName", "CurrentSeason"]].copy()
evaluation_sample["PrevLapTimeSeconds"] = X_test["PrevLapTimeSeconds"]
evaluation_sample["CurrentGridPosition"] = X_test["CurrentGridPosition"]
evaluation_sample["ActualLapTime"] = y_test
evaluation_sample["PredictedLapTime"] = y_pred
evaluation_sample.head()


Mean Absolute Error: 4.9019 seconds
R^2 Score: 0.5077


,Driver,EventName,CurrentSeason,PrevLapTimeSeconds,CurrentGridPosition,ActualLapTime,PredictedLapTime
753,ZHO,British Grand Prix,2024,98.976115,14,100.030700,95.823311
582,NOR,Bahrain Grand Prix,2024,102.606764,7,97.424561,100.406631
548,NOR,São Paulo Grand Prix,2023,82.711640,6,78.438971,82.562202
113,ALO,British Grand Prix,2022,95.567020,7,98.378100,96.652092
174,HAM,Italian Grand Prix,2022,87.689400,19,91.186623,86.240143
